# Статистические тесты для корреляций

Считаем корреляции Спирмена и Кендалла для всех методов из `histories/` и строим LaTeX-таблицы.

В нижней строке каждой таблицы — p-value парного t-теста Стьюдента (one-sided) с нулевой гипотезой
$H_0$: Graph $\le$ Best Baseline против альтернативы $H_1$: Graph $>$ Best Baseline.

P-value $< 0.05$ выделяются жирным.


In [1]:
import json

import numpy as np
from collections import defaultdict

from scipy.stats import spearmanr, kendalltau, ttest_rel


## Загрузка данных и подсчёт корреляций

Для каждой конфигурации графа (число итераций `n_iterations`) загружаем историю из `histories/` и для каждого эксперимента считаем корреляции предсказаний с истинными потерями.

Первые три модели (`Graph`, `FCN`, `Linear`) предсказывают потери напрямую, последние две (`L1`, `Molchanov`) — важности, которые инвертируются для согласования с направлением потерь (как в `paper_experiments_adv.ipynb`).


In [2]:
n_iterations_grid = [2, 5, 14, 30]

spearman_results = {}
kendall_results = {}

for n_iterations in n_iterations_grid:
    with open(f'histories/history_{n_iterations}_iterations.json', 'r') as json_file:
        history_to_save = json.loads(json_file.read())

    preds_results = history_to_save["preds_results"]
    y_true_all = history_to_save["y_true_all"]

    spearman_results[n_iterations] = defaultdict(list)
    kendall_results[n_iterations] = defaultdict(list)

    model_items = list(preds_results.items())

    for idx_experiment in range(len(y_true_all)):
        y_true = y_true_all[idx_experiment]

        # Модели, предсказывающие потери: Graph, FCN, Linear
        for model_name, all_preds in model_items[:3]:
            preds = all_preds[idx_experiment]

            spearman_results[n_iterations][model_name].append(spearmanr(preds, y_true)[0])
            kendall_results[n_iterations][model_name].append(kendalltau(preds, y_true)[0])

        # Методы на основе важности: L1, Molchanov (важность инвертируется)
        for model_name, all_preds in model_items[3:]:
            preds = np.asarray(all_preds[idx_experiment])
            preds = preds.max() - preds

            spearman_results[n_iterations][model_name].append(spearmanr(preds, y_true)[0])
            kendall_results[n_iterations][model_name].append(kendalltau(preds, y_true)[0])


## Парный t-тест Стьюдента

Для каждой колонки таблицы (конфигурации графа) определяем лучший бейзлайн — модель с наибольшим средним значением корреляции среди всех моделей, кроме `Graph`.

P-value считается парным t-тестом Стьюдента (пары — эксперименты с одинаковым индексом) с `alternative='greater'`, т.е. проверяется $H_0$: Graph $\le$ Best Baseline против $H_1$: Graph $>$ Best Baseline. Маленькое p-value означает, что `Graph` значимо лучше лучшего бейзлайна.


In [3]:
def get_best_baseline_name(results_by_model):
    baseline_means = {
        model_name: np.mean(values)
        for model_name, values in results_by_model.items()
        if model_name != 'Graph'
    }
    return max(baseline_means, key=baseline_means.get)


def get_pvalue(results_by_model):
    best_baseline_name = get_best_baseline_name(results_by_model)

    graph_values = np.asarray(results_by_model['Graph'])
    baseline_values = np.asarray(results_by_model[best_baseline_name])

    # H0: Graph <= Best Baseline, H1: Graph > Best Baseline
    return ttest_rel(graph_values, baseline_values, alternative='greater').pvalue


def format_pvalue(pvalue):
    pvalue_str = f"{pvalue:.3f}"
    if pvalue < 0.05:
        pvalue_str = rf"\textbf{{{pvalue_str}}}"
    return pvalue_str


## Генерация LaTeX-таблиц

Таблица как в `paper_experiments_adv.ipynb`: жирным выделяется лучший mean в колонке. Дополнительно в самый низ добавляется строка с p-value парного t-теста; p-value $< 0.05$ выделяются жирным.


In [15]:
def make_latex_table(results_by_group):
    group_names = list(results_by_group.keys())
    model_names = list(next(iter(results_by_group.values())).keys())

    # ищем лучший mean в каждом столбце
    best_mean = {}
    for group in group_names:
        best_mean[group] = max(
            np.mean(results_by_group[group][model])
            for model in model_names
        )

    lines = [
        r"\begin{tabular}{|l|" + "c|" * len(group_names) + r"}",
        r"\hline",
        r"\bfseries Model & "
        + " & ".join(group_names)
        + r" \\",
        r"\hline",
    ]

    for model in model_names:
        row = [("Taylor expansion" if model == "Molchanov" else model)]

        for group in group_names:
            values = results_by_group[group][model]

            mean = np.mean(values)
            std = np.std(values, ddof=1)

            value = f"{mean:.3f} $\\pm$ {std:.3f}"

            if mean < 0:
                value = f"${mean:.3f}$ $\\pm$ {std:.3f}"

            if np.isclose(mean, best_mean[group]):
                value = rf"\textbf{{{mean:.3f}}} $\pm$ {std:.3f}"

            row.append(value)

        lines.append(" & ".join(row) + r" \\")

    # строка с p-value парного t-теста H0: Graph <= Best Baseline
    pvalue_row = [r"p-value ($H_0$)"]
    for group in group_names:
        pvalue_row.append(format_pvalue(get_pvalue(results_by_group[group])))

    lines.append(" & ".join(pvalue_row) + r" \\")

    lines.extend([
        r"\hline",
        r"\end{tabular}",
    ])

    return "\n".join(lines)


## Таблица корреляций Спирмена


In [16]:
iterations2edges = {
    2: 15,
    5: 30,
    14: 75,
    30: 150
}

all_spearman = {
    f'{iterations2edges[n_iterations]} edges': results_by_model
    for n_iterations, results_by_model in spearman_results.items()
}

for group_name, results_by_model in all_spearman.items():
    print(f'{group_name}: best baseline — {get_best_baseline_name(results_by_model)}')

print()

latex_table = make_latex_table(all_spearman)
print(latex_table)


15 edges: best baseline — L1
30 edges: best baseline — L1
75 edges: best baseline — L1
150 edges: best baseline — Linear

\begin{tabular}{|l|c|c|c|c|}
\hline
\bfseries Model & 15 edges & 30 edges & 75 edges & 150 edges \\
\hline
Graph & 0.299 $\pm$ 0.235 & 0.296 $\pm$ 0.178 & 0.349 $\pm$ 0.128 & \textbf{0.411} $\pm$ 0.057 \\
FCN & $-0.009$ $\pm$ 0.213 & $-0.060$ $\pm$ 0.187 & $-0.028$ $\pm$ 0.090 & $-0.053$ $\pm$ 0.099 \\
Linear & 0.060 $\pm$ 0.187 & 0.201 $\pm$ 0.122 & 0.248 $\pm$ 0.090 & 0.294 $\pm$ 0.047 \\
L1 & \textbf{0.547} $\pm$ 0.063 & \textbf{0.505} $\pm$ 0.063 & \textbf{0.369} $\pm$ 0.060 & 0.280 $\pm$ 0.081 \\
Taylor expansion & 0.119 $\pm$ 0.247 & $-0.052$ $\pm$ 0.082 & $-0.136$ $\pm$ 0.076 & $-0.204$ $\pm$ 0.044 \\
p-value ($H_0$) & 0.991 & 0.996 & 0.670 & \textbf{0.000} \\
\hline
\end{tabular}


## Таблица корреляций Кендалла


In [17]:
all_kendall = {
    f'{iterations2edges[n_iterations]} edges': results_by_model
    for n_iterations, results_by_model in kendall_results.items()
}

for group_name, results_by_model in all_kendall.items():
    print(f'{group_name}: best baseline — {get_best_baseline_name(results_by_model)}')

print()

latex_table = make_latex_table(all_kendall)
print(latex_table)


15 edges: best baseline — L1
30 edges: best baseline — L1
75 edges: best baseline — L1
150 edges: best baseline — Linear

\begin{tabular}{|l|c|c|c|c|}
\hline
\bfseries Model & 15 edges & 30 edges & 75 edges & 150 edges \\
\hline
Graph & 0.253 $\pm$ 0.202 & 0.242 $\pm$ 0.146 & \textbf{0.282} $\pm$ 0.105 & \textbf{0.329} $\pm$ 0.045 \\
FCN & $-0.011$ $\pm$ 0.151 & $-0.042$ $\pm$ 0.128 & $-0.020$ $\pm$ 0.061 & $-0.036$ $\pm$ 0.066 \\
Linear & 0.039 $\pm$ 0.135 & 0.137 $\pm$ 0.083 & 0.167 $\pm$ 0.061 & 0.199 $\pm$ 0.032 \\
L1 & \textbf{0.401} $\pm$ 0.057 & \textbf{0.353} $\pm$ 0.049 & 0.259 $\pm$ 0.041 & 0.192 $\pm$ 0.057 \\
Taylor expansion & 0.094 $\pm$ 0.182 & $-0.033$ $\pm$ 0.055 & $-0.090$ $\pm$ 0.051 & $-0.136$ $\pm$ 0.030 \\
p-value ($H_0$) & 0.959 & 0.976 & 0.269 & \textbf{0.000} \\
\hline
\end{tabular}
